<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    7.lab. End-to-End Cancer Classification API
  </strong>
</div>

### Background

You will build an **end-to-end** solution for the Wisconsin Diagnostic Cancer
Dataset, turning a classifier into a production-ready REST API that labels
tissue samples as **malignant** or **benign**.

The dataset includes:

- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- Binary classification: **Malignant** or **Benign**

### Description

Your task is to:

- Evaluate multiple classification approaches (SVM, Neural Networks, etc.)
- Optimize the chosen model for accuracy and inference speed
- Create a REST API endpoint that accepts image-derived data and returns predictions
- Ensure inference stays consistently below 100 ms
- Document the model selection process and performance metrics

### Deliverables

- This notebook documents the evaluation, selection and optimization process.
- The FastAPI service lives in `src/` and is documented in `README.md`.
- Model comparison metrics are written to `data/model_comparison.csv`.
- Latency benchmarks are written to `data/latency_benchmark.csv`.
- The trained model is persisted to `models/cancer_classifier.joblib`.

> **Note:** the Wisconsin dataset ships the 30 **image-derived features** (not raw
> pixels), so the API accepts feature vectors, exactly the inputs a feature
> extractor would produce from a tissue image. The trained pipeline loads from
> disk once and is reused across requests.

## 1. Load and Explore the Dataset

In [ ]:
import json
import pickle
import platform
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

BASE_DIR = Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_FEATURES = 30
MALIGNANT_LABEL = 0
LATENCY_BUDGET_MS = 100.0

np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__}")
print(f"Working directory: {BASE_DIR}")

In [ ]:
raw = load_breast_cancer(as_frame=True)
X = raw.data
y = raw.target
feature_names = list(raw.feature_names)
X_array = X.to_numpy(dtype=np.float64)
y_array = y.to_numpy()

print(f"Instances: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Classes: {dict(zip(raw.target_names, np.bincount(y)))}")
print(f"Malignant (label {MALIGNANT_LABEL}): {(y == MALIGNANT_LABEL).sum()}")
X.head()

## 2. Candidate Models

We compare six families that are representative of the trade-off between
accuracy and inference cost:

| Model | Why it is interesting |
|-------|-----------------------|
| Logistic Regression | Strong linear baseline, natively calibrated probabilities, tiny and fast |
| Linear SVM | Maximum-margin linear separator |
| RBF SVM | Non-linear kernel model from the previous lab |
| Random Forest | Bagged trees, robust but heavier |
| Gradient Boosting | Strong tabular learner, moderate cost |
| Neural Network (MLP) | Non-linear learner with 32-16 hidden units |

Every model that benefits from feature scaling is wrapped in a
`StandardScaler` pipeline so the exported artifact is self-contained.

In [ ]:
candidates = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    ),
    "Linear SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(random_state=RANDOM_STATE)),
        ]
    ),
    "RBF SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "Neural Network (MLP)": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                MLPClassifier(
                    hidden_layer_sizes=(32, 16),
                    max_iter=3000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}
list(candidates)

## 3. Evaluation Protocol

To avoid drawing conclusions from a single lucky split, every candidate is
scored with **repeated stratified 5-fold cross-validation** (5 folds x 5
repetitions = 25 fits) over the whole dataset.

Besides the usual classification metrics we measure:

- **cv_f1_malignant**: F1 for the *malignant* class. In a screening context a
  false negative is the most dangerous error, so this is the primary metric.
- **p50_ms / p95_ms**: single-sample `predict_proba` latency, measured after a
  warm-up phase so that JIT/CPU caching does not distort the numbers.
- **size_kb**: serialized artifact size, a proxy for deployment cost.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "f1_malignant": make_scorer(f1_score, pos_label=MALIGNANT_LABEL),
    "roc_auc": "roc_auc",
}


def invoke(model, matrix):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(matrix)
    return model.decision_function(matrix)


def measure_latency(model, sample, iterations=400, warmup=20):
    matrix = np.asarray(sample, dtype=np.float64).reshape(1, -1)
    for _ in range(warmup):
        invoke(model, matrix)
    latencies = np.empty(iterations)
    for index in range(iterations):
        start = time.perf_counter()
        invoke(model, matrix)
        latencies[index] = (time.perf_counter() - start) * 1000.0
    return latencies

In [ ]:
rows = []
sample = X_array[0]

for name, model in candidates.items():
    cv_result = cross_validate(model, X_array, y_array, cv=cv, scoring=scoring, n_jobs=-1)
    model.fit(X_array, y_array)
    latencies = measure_latency(model, sample)
    rows.append(
        {
            "model": name,
            "cv_accuracy": cv_result["test_accuracy"].mean(),
            "cv_accuracy_std": cv_result["test_accuracy"].std(),
            "cv_f1_malignant": cv_result["test_f1_malignant"].mean(),
            "cv_f1_malignant_std": cv_result["test_f1_malignant"].std(),
            "cv_roc_auc": cv_result["test_roc_auc"].mean(),
            "p50_ms": float(np.median(latencies)),
            "p95_ms": float(np.percentile(latencies, 95)),
            "size_kb": len(pickle.dumps(model)) / 1024,
        }
    )
    print(f"evaluated: {name}")

comparison = (
    pd.DataFrame(rows)
    .sort_values("cv_f1_malignant", ascending=False)
    .reset_index(drop=True)
)
comparison.round(4)

In [ ]:
comparison.round(4).to_csv(DATA_DIR / "model_comparison.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
order = comparison["model"]

axes[0].barh(
    order,
    comparison["cv_f1_malignant"],
    xerr=comparison["cv_f1_malignant_std"],
    color="#0e3262",
)
axes[0].set_xlabel("F1 (malignant)")
axes[0].set_title("Cross-validated F1 (malignant)")
axes[0].invert_yaxis()
axes[0].grid(True, axis="x", alpha=0.3)

axes[1].barh(order, comparison["p95_ms"], color="#c0392b")
axes[1].axvline(LATENCY_BUDGET_MS, color="black", linestyle="--", label="100 ms budget")
axes[1].set_xlabel("p95 latency (ms)")
axes[1].set_title("Single-sample inference latency")
axes[1].invert_yaxis()
axes[1].legend()
axes[1].grid(True, axis="x", alpha=0.3)

axes[2].barh(order, comparison["size_kb"], color="#27ae60")
axes[2].set_xlabel("Serialized size (KB)")
axes[2].set_title("Model artifact size")
axes[2].invert_yaxis()
axes[2].grid(True, axis="x", alpha=0.3)

plt.suptitle("Wisconsin Cancer — candidate model comparison", fontsize=14)
plt.tight_layout()
plt.savefig(DATA_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

comparison.round(4)

## 4. Model Selection

The selection rule is deliberately simple and stated in advance:

1. **Hard constraint:** the p95 inference latency must stay below the 100 ms
   budget.
2. **Primary metric:** the highest cross-validated F1 for the malignant class.
3. **Tie-break:** models whose F1 is within `0.005` of the best are considered
   statistically tied; among them we prefer the smallest artifact, then the
   lowest latency. This favours simpler, cheaper deployments when the metrics
   are indistinguishable.

The difference between the top models is small (well under one percentage
point), which is expected for a small, clean tabular dataset. In cases like
this, model size, latency and probability calibration become the deciding
factors rather than a fraction of a point of accuracy.

In [ ]:
TIE_TOLERANCE = 0.005
eligible = comparison[comparison["p95_ms"] < LATENCY_BUDGET_MS].copy()
best_f1 = eligible["cv_f1_malignant"].max()
contenders = eligible[eligible["cv_f1_malignant"] >= best_f1 - TIE_TOLERANCE]
selected = contenders.sort_values(["size_kb", "p95_ms"]).iloc[0]
selected_name = str(selected["model"])

print(f"Best cross-validated F1 (malignant): {best_f1:.4f}")
print(f"Statistical contenders (within {TIE_TOLERANCE}): {list(contenders['model'])}")
print(f"Selected model: {selected_name}")

## 5. Hyperparameter Optimization

The selected model is tuned with a grid search using the same repeated
cross-validation and the malignant F1 as the objective. For a linear model the
only meaningful knob is the regularisation strength `C`: it controls the
bias/variance trade-off and, indirectly, the magnitude of the coefficients
(which keeps inference fast because nothing else changes).

In [ ]:
param_grid = {"clf__C": [0.1, 0.5, 1.0, 10.0]}
search = GridSearchCV(
    candidates[selected_name],
    param_grid,
    cv=cv,
    scoring=make_scorer(f1_score, pos_label=MALIGNANT_LABEL),
    n_jobs=-1,
)
search.fit(X_array, y_array)

print(f"Best hyperparameters: {search.best_params_}")
print(f"Best cross-validated F1 (malignant): {search.best_score_:.4f}")
best_model = search.best_estimator_

## 6. Final Evaluation

We inspect the tuned model through **out-of-fold predictions**: every sample is
classified by a model that did not see it during training. This gives an honest
confusion matrix and ROC curve on the full dataset instead of a single split.

In [ ]:
oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_predictions = cross_val_predict(best_model, X_array, y_array, cv=oof_cv, method="predict")
oof_probabilities = cross_val_predict(
    best_model, X_array, y_array, cv=oof_cv, method="predict_proba"
)[:, MALIGNANT_LABEL]

print(classification_report(y_array, oof_predictions, target_names=raw.target_names, digits=4))

oof_roc_auc = roc_auc_score((y_array == MALIGNANT_LABEL).astype(int), oof_probabilities)
print(f"Out-of-fold ROC-AUC (malignant as positive): {oof_roc_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_array, oof_predictions, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=raw.target_names).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Out-of-fold confusion matrix")

fpr, tpr, _ = roc_curve((y_array == MALIGNANT_LABEL).astype(int), oof_probabilities)
axes[1].plot(fpr, tpr, color="#0e3262", label=f"AUC = {oof_roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve (malignant as positive)")
axes[1].legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best_cv = cross_validate(best_model, X_array, y_array, cv=cv, scoring=scoring, n_jobs=-1)
metrics = {
    "cv_accuracy": float(best_cv["test_accuracy"].mean()),
    "cv_accuracy_std": float(best_cv["test_accuracy"].std()),
    "cv_f1_malignant": float(best_cv["test_f1_malignant"].mean()),
    "cv_f1_malignant_std": float(best_cv["test_f1_malignant"].std()),
    "cv_roc_auc": float(best_cv["test_roc_auc"].mean()),
    "oof_accuracy": float(accuracy_score(y_array, oof_predictions)),
    "oof_f1_malignant": float(f1_score(y_array, oof_predictions, pos_label=MALIGNANT_LABEL)),
    "oof_roc_auc": float(oof_roc_auc),
}
pd.Series(metrics).round(4)

## 7. Persist the Deployed Model

The API loads a single artifact from disk, so we now refit the tuned pipeline on
**all** available samples and write it together with its metadata. Refitting on
the full dataset maximizes the data used by the production model; the retraining
slightly changes the out-of-fold estimates above, which is why both the CV and
the artifact are versioned together.

In [ ]:
best_model.fit(X_array, y_array)
joblib.dump(best_model, MODELS_DIR / "cancer_classifier.joblib")

steps = " -> ".join(best_model.named_steps) if hasattr(best_model, "named_steps") else selected_name
metadata = {
    "model_version": "1.0.0",
    "algorithm": selected_name,
    "pipeline": steps,
    "trained_on": "sklearn.datasets.load_breast_cancer (569 samples, 30 features)",
    "trained_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "sklearn_version": sklearn.__version__,
    "feature_names": feature_names,
    "classes": [int(label) for label in best_model.classes_],
    "class_labels": {"0": "malignant", "1": "benign"},
    "hyperparameters": {
        key: value if isinstance(value, (int, float, str)) else str(value)
        for key, value in search.best_params_.items()
    },
    "metrics": metrics,
}
(MODELS_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))

print(f"Saved model: {MODELS_DIR / 'cancer_classifier.joblib'}")
print(json.dumps(metrics, indent=2))

## 8. Inference Latency Under Concurrent Requests

A production API rarely serves one request at a time. We benchmark two paths
across increasing levels of concurrency:

- **model (direct):** the model called in-process from a thread pool, isolating
  pure inference cost.
- **api (/predict):** full HTTP round-trip through the FastAPI application
  (validation, JSON serialisation, endpoint), driven with `httpx.ASGITransport`.

Because the prediction is CPU-bound and executed in a worker thread (the
endpoint is a synchronous `def`, so FastAPI offloads it from the event loop),
latency grows and throughput plateaus as workers compete for the GIL. The key
question is whether the p95 stays inside the 100 ms budget.

In [ ]:
def benchmark_direct(model, sample, concurrency, requests_per_worker=200):
    matrix = np.asarray(sample, dtype=np.float64).reshape(1, -1)
    for _ in range(20):
        model.predict_proba(matrix)

    def worker(_):
        latencies = []
        for _ in range(requests_per_worker):
            start = time.perf_counter()
            model.predict_proba(matrix)
            latencies.append((time.perf_counter() - start) * 1000.0)
        return latencies

    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        batches = list(pool.map(worker, range(concurrency)))
    wall = time.perf_counter() - start

    latencies = np.concatenate(batches)
    total = latencies.size
    return {
        "concurrency": concurrency,
        "requests": total,
        "throughput_rps": total / wall,
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
    }

In [ ]:
import asyncio

import httpx

from src.app import app


async def benchmark_api(sample, concurrency, requests_per_worker=100):
    payload = {"features": [float(value) for value in sample]}
    transport = httpx.ASGITransport(app=app)

    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:

        async def worker():
            latencies = []
            for _ in range(requests_per_worker):
                start = time.perf_counter()
                response = await client.post("/predict", json=payload)
                response.raise_for_status()
                latencies.append((time.perf_counter() - start) * 1000.0)
            return latencies

        start = time.perf_counter()
        batches = await asyncio.gather(*(worker() for _ in range(concurrency)))
        wall = time.perf_counter() - start

    latencies = np.concatenate(batches)
    total = latencies.size
    return {
        "concurrency": concurrency,
        "requests": total,
        "throughput_rps": total / wall,
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
    }


async def run_api_benchmark(sample, levels):
    return [await benchmark_api(sample, level) for level in levels]


CONCURRENCY_LEVELS = [1, 2, 4, 8, 16]

direct_bench = pd.DataFrame(
    benchmark_direct(best_model, sample, level) for level in CONCURRENCY_LEVELS
)
direct_bench["path"] = "model (direct)"

api_bench = pd.DataFrame(await run_api_benchmark(sample, CONCURRENCY_LEVELS))
api_bench["path"] = "api (/predict)"

benchmark = pd.concat([direct_bench, api_bench], ignore_index=True)
benchmark.to_csv(DATA_DIR / "latency_benchmark.csv", index=False)
benchmark.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for path, group in benchmark.groupby("path"):
    axes[0].plot(group["concurrency"], group["p95_ms"], marker="o", label=path)
    axes[1].plot(group["concurrency"], group["throughput_rps"], marker="o", label=path)

axes[0].axhline(LATENCY_BUDGET_MS, color="black", linestyle="--", label="100 ms budget")
axes[0].set_xlabel("Concurrent workers")
axes[0].set_ylabel("p95 latency (ms)")
axes[0].set_title("Latency under concurrency")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Concurrent workers")
axes[1].set_ylabel("Throughput (requests/s)")
axes[1].set_title("Throughput under concurrency")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Inference performance — direct model vs HTTP API", fontsize=14)
plt.tight_layout()
plt.savefig(DATA_DIR / "latency_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Discussion

### 9.1 Desempenho dos modelos avaliados

Os seis modelos ficaram concentrados numa faixa estreita de acurácia e F1, o
que é típico desse dataset: as 30 features derivadas da imagem são limpas,
bem escaladas e com alta separabilidade entre as classes. Como já tínhamos
visto no lab de PCA/K-means, uma única direção linear já separa a maior parte
das amostras, então modelos lineares conseguem acompanhar modelos não lineares
sem esforço extra.

O `Random Forest` foi o mais fraco em F1 para a classe maligna e, de longe, o
mais caro: usa 300 árvores e o artefato fica na casa de centenas de KB, com
latência p95 ordens de magnitude acima dos modelos lineares. O `Gradient
Boosting` ficou no meio, com desempenho bom mas custo de inferência
intermediário. A `MLP` (rede neural 32-16) chegou perto dos melhores em
média, porém com mais variância entre folds, o que indica menor estabilidade
num dataset desse tamanho.

### 9.2 Por que a Regressão Logística foi escolhida

A `Logistic Regression` ficou no topo da validação cruzada repetida para F1 da
classe maligna e para acurácia, e está estatisticamente empatada com o SVM RBF
e a MLP (diferença bem abaixo de 1 ponto percentual). Quando o desempenho é
indistinguível, os critérios de engenharia passam a decidir:

- **Tamanho:** o artefato é o menor de todos (poucos KB), enquanto SVM e MLP
  são uma ordem de magnitude maiores.
- **Latência:** `predict_proba` é uma única multiplicação matricial mais a
  sigmoide, então a inferência fica na casa de microssegundos a poucos
  milissegundos.
- **Probabilidades calibradas de graça:** a saída já é uma probabilidade
  nativa (sigmoide), sem `Platt scaling`. Isso importa porque a API devolve
  `malignant_probability` e `confidence`.
- **Determinismo:** o treino de uma regressão logística não depende de
  inicialização aleatória, então o artefato é perfeitamente reproduzível.

O SVM RBF ficou praticamente empatado e seria a escolha se o problema fosse
claramente não linear. A regressão logística é o modelo honesto para este
dataset: tão preciso quanto, mais simples e mais barato de servir.

### 9.3 Otimizações aplicadas

- **Padronização dentro do pipeline.** O `StandardScaler` faz parte do
  artefato salvo, então treino e inferência usam exatamente a mesma
  transformação, sem risco de esquecer o escalonamento no servidor.
- **Busca de hiperparâmetro.** O `GridSearchCV` sobre `C` confirmou que o valor
  padrão já era ótimo sob validação cruzada repetida; variar `C` demais piora
  a generalização (regularização fraca demais aproxima demais dos dados de
  treino).
- **Modelo carregado uma única vez.** A API usa um *singleton* em cache: o
  `joblib.load` acontece na primeira requisição e o mesmo objeto é reutilizado
  depois, evitando ler o arquivo a cada chamada.
- **Endpoint síncrono.** As rotas são funções `def`; o FastAPI as executa num
  *thread pool*, então a predição CPU-bound não bloqueia o event loop e é
  possível atender várias requisições em paralelo.
- **Batch.** O endpoint `/predict/batch` faz uma única chamada vetorizada de
  `predict_proba`, amortizando o overhead por amostra.

### 9.4 Comportamento sob concorrência

Nosso benchmark mostra que, com 1 worker, a latência ponta a ponta fica na
casa de poucos milissegundos. Conforme a concorrência sobe, a latência cresce e
a vazão satura — comportamento esperado para carga CPU-bound em Python, já que
as predições disputam o GIL depois de sairem do pool de threads. O ponto
importante é que mesmo nas concorrências mais altas o p95 permanece bem abaixo
do orçamento de 100 ms, porque cada inferência é barata (dezenas de
microssegundos). Em produção, o caminho para escalar vazão seria rodar
múltiplos *workers* do uvicorn (um processo por núcleo de CPU), cada um com sua
própria cópia do modelo.

### 9.5 Limitações

- O dataset entrega as 30 features já extraídas da imagem, não os pixels. A API
  recebe esses vetores; uma classificação de imagem de ponta a ponta exigiria
  um extrator de features antes da inferência.
- As métricas vêm de validação cruzada sobre um único dataset, sem divisão
  externa independente. Num cenário clínico real seria necessário validar em
  dados de outra instituição.
- A `Regressão Logística` assume fronteira linear; casos ambíguos (que caem na
  zona de sobreposição entre as classes) continuam sendo o principal erro,
  independentemente do modelo.